In [1]:
#Библиотеки
import cv2
import pdb
import numpy as np
import time
import os
from numpy.lib.stride_tricks import sliding_window_view

In [2]:
#Сохранение изображения
def SaveImage(image, original_path, change):
    directory = os.path.dirname(original_path)
    
    basename = os.path.splitext(os.path.basename(original_path))[0]
    
    new_filename = f"{basename}{change}.jpg"
    new_path = os.path.join(directory, new_filename)

    cv2.imwrite(new_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    print(f"Сохранено: {new_filename}")
    
    return new_path

In [3]:
#Преобразование в numpy массив
def ConvertNumpy(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Не удалось загрузить изображение")
        return None
    
    rgb_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"Изображение загружено")
    return rgb_image

In [4]:
#Приведение цветного изображения к полутоновому (собственная реализация)
def MyGrayscale(image):
    height, width = image.shape[0], image.shape[1]
    gray = np.zeros((height, width), dtype=np.float32)
    
    gray = 0.299 * image[:, :, 0] + 0.587 * image[:, :, 1] + 0.114 * image[:, :, 2]
    
    return gray.astype(np.uint8)

In [5]:
#Приведение цветного изображения к полутоновому (библиотечная реализация реализация)
def LibraryGrayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

In [6]:
#Cвёртка c использованием двумерной маски (собственная реализация)
def MyConvolution(image, kernel):
    if len(image.shape) == 3:
        h, w, c = image.shape
        kh, kw = kernel.shape
        pad_h, pad_w = kh // 2, kw // 2
        
        result_channels = []
        for channel in range(c):
            padded = np.pad(image[:,:,channel], ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
            windows = sliding_window_view(padded, (kh, kw))
            channel_result = np.sum(windows * kernel.reshape(1, 1, kh, kw), axis=(2, 3))
            channel_result = np.clip(channel_result, 0, 255)
            result_channels.append(channel_result)
            
        result = np.stack(result_channels, axis=-1)
        return result.astype(np.uint8)
    else:
        h, w = image.shape
        kh, kw = kernel.shape
        pad_h, pad_w = kh // 2, kw // 2
        
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
        windows = sliding_window_view(padded, (kh, kw))      
        result = np.sum(windows * kernel.reshape(1, 1, kh, kw), axis=(2, 3))
        result = np.clip(result, 0, 255)
        
        return result.astype(np.uint8)

In [7]:
#Cвёртка c использованием двумерной маски (библиотечная реализация)
def LibraryConvolution(image, kernel):
    if len(image.shape) == 2:
        return cv2.filter2D(image, -1, kernel)
    else:
        image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        result_bgr = cv2.filter2D(image_bgr, -1, kernel)
        return cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

In [8]:
#Cглаживание (применение оператора Гаусса) (Собственная реализация)
def MyBlur(image, size=5, sigma=1.0):
    kernel = np.zeros((size, size), dtype=np.float32)
    center = size // 2
    
    x = np.arange(size) - center
    x = np.stack([x]*size)
    y = np.matrix.transpose(x)
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    
    kernel = kernel / np.sum(kernel)
    return MyConvolution(image, kernel)

In [9]:
#Cглаживание (применение оператора Гаусса) (библиотечная реализация)
def LibraryBlur(image, size=5, sigma=1):
    if len(image.shape) == 2:
        return cv2.GaussianBlur(image, (size, size), sigma)
    else:
        image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        result_bgr = cv2.GaussianBlur(image_bgr, (size, size), sigma)
        return cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

In [10]:
#Выделение] границ (например, применение оператора Собеля) (собственная реализация)
def MySobel(image):
    kernel_x = np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)
    
    kernel_y = np.array([[-1, -2, -1],
                         [0, 0, 0],
                         [1, 2, 1]], dtype=np.float32)
    
    grad_x = MyConvolution(image, kernel_x)
    grad_y = MyConvolution(image, kernel_y)
    
    approximate_value = np.sqrt(grad_x.astype(np.float32)**2 + grad_y.astype(np.float32)**2)
    approximate_value = np.clip(approximate_value, 0, 255)
    
    return approximate_value.astype(np.uint8)

In [11]:
#Выделение границ (например, применение оператора Собеля) (библиотечная реализация)
def LibrarySobel(image):
    grad_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    
    approximate_value = np.sqrt(grad_x**2 + grad_y**2)
    approximate_value = np.clip(approximate_value, 0, 255)
    
    return approximate_value.astype(np.uint8)

In [12]:
#Расчёт времени выполнения и сохранение
def ImageProcessing(image_path):
    rgb_image = ConvertNumpy(image_path)

    #Ядро теснения
    kernel = np.array([[-2, -1, 0],
                        [-1, 1, 1],
                        [0, 1, 2]], dtype=np.float32)
    
    # 1 задание
    print("\nПриведение цветного изображения к полутоновому:")
    
    start = time.time()
    my_grayscale = MyGrayscale(rgb_image)
    my_time = time.time() - start
    SaveImage(np.stack([my_grayscale]*3, axis=-1), image_path, '_my_grayscale')
    print(f"Время собственной реализация: {my_time:.4f} сек")
    
    start = time.time()
    lib_grayscale = LibraryGrayscale(rgb_image)
    lib_time = time.time() - start
    SaveImage(np.stack([lib_grayscale]*3, axis=-1), image_path, '_lib_grayscale')
    print(f"Время библиотечной реализации: {lib_time:.4f} сек")
    print(f"Разница: {abs(my_time - lib_time):.4f} сек")

    # 2 задание
    print("\nCвёртка c использованием двумерной маски:")
    
    start = time.time()
    my_convolution = MyConvolution(rgb_image, kernel)
    my_time = time.time() - start
    SaveImage(my_convolution, image_path, '_my_conv')
    print(f"Время собственной реализация: {my_time:.4f} сек")
    
    start = time.time()
    lib_convolution = LibraryConvolution(rgb_image, kernel)
    lib_time = time.time() - start
    SaveImage(lib_convolution, image_path, '_lib_conv')
    print(f"Время библиотечной реализации: {lib_time:.4f} сек")
    print(f"Разница: {abs(my_time - lib_time):.4f} сек")

    # 3 задание
    print("\nCглаживание (применение оператора Гаусса)")
    
    start = time.time()
    my_blur = MyBlur(rgb_image, 5, 1.0)
    my_time = time.time() - start
    SaveImage(my_blur, image_path, '_my_blur')
    print(f"Время собственной реализация: {my_time:.4f} сек")
    
    start = time.time()
    lib_blur = LibraryBlur(my_grayscale, 5, 1.0)
    lib_time = time.time() - start
    SaveImage(lib_blur, image_path, '_lib_blur')
    print(f"Время библиотечной реализации: {lib_time:.4f} сек")
    print(f"Разница: {abs(my_time - lib_time):.4f} сек")

    # 4 задание
    print("\nВыделение границ (применение оператора Собеля):")
    
    start = time.time()
    my_sobel_result = MySobel(my_grayscale)
    my_time = time.time() - start
    SaveImage(np.stack([my_sobel_result]*3, axis=-1), image_path, '_my_sobel')
    print(f"Время собственной реализация: {my_time:.4f} сек")
    
    start = time.time()
    lib_sobel_result = LibrarySobel(my_grayscale)
    lib_time = time.time() - start
    SaveImage(np.stack([lib_sobel_result]*3, axis=-1), image_path, '_lib_sobel')
    print(f"Время библиотечной реализации: {lib_time:.4f} сек")
    print(f"Разница: {abs(my_time - lib_time):.4f} сек")

In [13]:
if __name__ == "__main__":
    input_dir = "paintings"
    image_path = "paintings/78143.jpg"

    #Проверка файла
    if os.path.exists(image_path):
        ImageProcessing(image_path)
    else:
        print(f"Файл не найден: {image_path}")

Изображение загружено

Приведение цветного изображения к полутоновому:
Сохранено: 78143_my_grayscale.jpg
Время собственной реализация: 0.1756 сек
Сохранено: 78143_lib_grayscale.jpg
Время библиотечной реализации: 0.0043 сек
Разница: 0.1713 сек

Cвёртка c использованием двумерной маски:
Сохранено: 78143_my_conv.jpg
Время собственной реализация: 4.3714 сек
Сохранено: 78143_lib_conv.jpg
Время библиотечной реализации: 0.0726 сек
Разница: 4.2988 сек

Cглаживание (применение оператора Гаусса)
Сохранено: 78143_my_blur.jpg
Время собственной реализация: 10.1474 сек
Сохранено: 78143_lib_blur.jpg
Время библиотечной реализации: 0.0037 сек
Разница: 10.1437 сек

Выделение границ (применение оператора Собеля):
Сохранено: 78143_my_sobel.jpg
Время собственной реализация: 2.6206 сек
Сохранено: 78143_lib_sobel.jpg
Время библиотечной реализации: 0.2761 сек
Разница: 2.3445 сек
